# Large language models: a hands-on implementation

**Track:** Language models · **Guide:** [`036-llm.md`](../guides/concepts/036-llm.md)

An LLM is a language model with enough learned parameters and data to handle many language tasks through prompting.

> **Intuition:** A broad, compressed library of language patterns that still needs a task, context, and verification.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Large language models]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Design a support assistant with a finite context window, bounded output, and a fallback for uncertain answers.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Train a tiny language model

A language model learns a probability distribution over the next token. A smoothed bigram table is small enough to inspect.


In [ ]:
from collections import Counter, defaultdict
from itertools import pairwise

training_text = [
    "retrieval finds useful evidence",
    "language models generate text",
    "retrieval helps language models",
    "evidence supports grounded answers",
]
bigram_counts = defaultdict(Counter)
unigram_counts = Counter()
for sentence in training_text:
    words = ["<BOS>", *sentence.split(), "<EOS>"]
    unigram_counts.update(words[:-1])
    for left, right in pairwise(words):
        bigram_counts[left][right] += 1
vocabulary = sorted(set(unigram_counts) | {"<EOS>"})
print("vocabulary size:", len(vocabulary))

## 2. Compute next-token probabilities

Additive smoothing assigns nonzero probability to unseen transitions; neural LLMs use far richer learned representations.


In [ ]:
def next_token_distribution(context, alpha=0.2):
    current = context.split()[-1] if context.split() else "<BOS>"
    counts = bigram_counts[current]
    denominator = sum(counts.values()) + alpha * len(vocabulary)
    return {
        token: (counts[token] + alpha) / denominator
        for token in vocabulary
    }

distribution = next_token_distribution("language models")
print(sorted(distribution.items(), key=lambda item: -item[1])[:5])

## 3. Generate with a bounded context window

Generation repeatedly samples/chooses a next token. Bound output length and handle an end token; production LLMs also enforce token and tool limits.


In [ ]:
import random


def generate_text(prompt, max_tokens=10, seed=3):
    rng = random.Random(seed)
    words = prompt.split()
    for _ in range(max_tokens):
        distribution = next_token_distribution(" ".join(words))
        tokens, probabilities = zip(*distribution.items())
        token = rng.choices(tokens, weights=probabilities, k=1)[0]
        if token == "<EOS>":
            break
        words.append(token)
    return " ".join(words)

print(generate_text("retrieval helps", max_tokens=8))

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/036-llm.md) and the
[learning roadmap](../guides/ROADMAP.md).
